# Sprint 4 — Webinar 14 (Práctica)
**Modalidad:** Coding Together + Breakout Rooms
**Tema:** Data Journey → Funnels → Cohorts — Ejercicios aplicados en SQL
**Motor SQL:** [SQL Workbench](https://sql-workbench.com/) (DuckDB en el navegador)

> Esta sesión práctica aplica lo aprendido en **S4W13 (Teórico)**. Usaremos un **nuevo escenario** — una plataforma de streaming de música — para practicar CTEs, funnels y cohorts en un contexto diferente al e-commerce.

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Estructura y timeboxing

| Tiempo | Bloque | Modalidad | Contenido |
|-------:|--------|-----------|-----------|
| 0–10 | Bloque 0 | Breakout Rooms | Calentamiento: exploración de datos y validación básica |
| 10–35 | Bloque 1 | Coding Together | CTEs aplicadas: desde lo básico hasta encadenamiento |
| 35–60 | Bloque 2 | Coding Together | Funnel de conversión de la plataforma de streaming |
| 60–80 | Bloque 3 | Coding Together | Cohorts y retención semanal |
| 80–95 | Bloque 4 | Breakout Rooms | Ejercicio integrador: análisis completo del journey |
| 95–100 | Cierre | Plenaria | Discusión de hallazgos, takeaways y próximos pasos |

> **Recomendación:** Abre [sql-workbench.com](https://sql-workbench.com/) y ejecuta el Bloque 0 (DDL + Seed) antes de arrancar.

## Objetivos de la sesión

Al finalizar esta práctica, el/la estudiante podrá:

1. **Explorar** un dataset de eventos de una plataforma de streaming y validar su calidad.
2. **Escribir CTEs** encadenadas para transformar datos paso a paso.
3. **Construir** un funnel de conversión completo (con drop-off y tasas de conversión).
4. **Calcular** tiempos entre etapas del journey para identificar cuellos de botella.
5. **Armar** una tabla de retención por cohort mensual y segmentarla por atributos.

## 🎵 Escenario: MusicFlow — Plataforma de Streaming

**MusicFlow** es una plataforma de streaming de música que acaba de lanzar una versión renovada de su app. El equipo de producto quiere entender:

- ¿Cómo es el **journey** de un usuario nuevo desde que se registra hasta que se convierte en suscriptor premium?
- ¿Dónde están los **cuellos de botella** en el funnel de conversión?
- ¿Qué tan bien **retienen** a los usuarios las diferentes cohortes de registro?
- ¿Los usuarios de diferentes **canales** o **planes** se comportan diferente?

### Journey de MusicFlow

| Etapa | Evento | Significado |
|-------|--------|-------------|
| 1. Llegada | `app_open` | El usuario abre la app |
| 2. Exploración | `search_song` | Busca una canción o artista |
| 3. Consumo | `play_song` | Reproduce una canción |
| 4. Engagement | `create_playlist` | Crea una playlist personalizada |
| 5. Conversión | `subscribe_premium` | Se suscribe al plan premium |

> El equipo de MusicFlow necesita tu ayuda para analizar los datos del **Q1 2024** (enero–marzo).

---
# Bloque 0 · Preparación del entorno

> **Ejecuta estos dos bloques de SQL en [sql-workbench.com](https://sql-workbench.com/) antes de empezar.** Crean las tablas y cargan los datos del escenario de MusicFlow.

### 0.1 Creación de tablas (DDL)

```sql
-- =============================================
-- DDL: MusicFlow — Plataforma de Streaming
-- Compatible con DuckDB (SQL Workbench)
-- =============================================

CREATE OR REPLACE TABLE users (
    user_id     INTEGER PRIMARY KEY,
    signup_ts   TIMESTAMP NOT NULL,
    plan        VARCHAR NOT NULL,       -- 'free' o 'premium'
    channel     VARCHAR NOT NULL,       -- 'organic', 'ads', 'referral', 'social'
    device      VARCHAR NOT NULL,       -- 'mobile', 'desktop', 'tablet'
    country     VARCHAR NOT NULL        -- 'CO', 'MX', 'AR', 'PE', 'CL'
);

CREATE OR REPLACE TABLE events (
    event_id    INTEGER PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_name  VARCHAR NOT NULL,
    event_ts    TIMESTAMP NOT NULL,
    props       VARCHAR NOT NULL DEFAULT '{}'
);
```

### 0.2 Carga de datos (Seed)

```sql
-- =============================================
-- SEED: Usuarios de MusicFlow (Q1 2024)
-- 20 usuarios distribuidos en 3 cohortes mensuales
-- =============================================

INSERT INTO users (user_id, signup_ts, plan, channel, device, country) VALUES
    -- Cohorte Enero 2024 (7 usuarios)
    (1,  '2024-01-03 08:30:00', 'free',    'organic',  'mobile',  'CO'),
    (2,  '2024-01-05 14:15:00', 'free',    'ads',      'desktop', 'MX'),
    (3,  '2024-01-08 10:00:00', 'free',    'referral', 'mobile',  'AR'),
    (4,  '2024-01-12 19:20:00', 'free',    'social',   'mobile',  'CO'),
    (5,  '2024-01-15 11:45:00', 'free',    'organic',  'tablet',  'PE'),
    (6,  '2024-01-20 16:30:00', 'free',    'ads',      'mobile',  'MX'),
    (7,  '2024-01-25 09:00:00', 'free',    'social',   'desktop', 'CL'),

    -- Cohorte Febrero 2024 (7 usuarios)
    (8,  '2024-02-02 07:45:00', 'free',    'organic',  'mobile',  'CO'),
    (9,  '2024-02-06 13:30:00', 'free',    'ads',      'mobile',  'MX'),
    (10, '2024-02-10 18:00:00', 'free',    'referral', 'desktop', 'AR'),
    (11, '2024-02-14 22:10:00', 'free',    'social',   'mobile',  'PE'),
    (12, '2024-02-18 10:15:00', 'free',    'organic',  'tablet',  'CO'),
    (13, '2024-02-22 15:00:00', 'free',    'ads',      'mobile',  'CL'),
    (14, '2024-02-26 08:30:00', 'free',    'referral', 'mobile',  'MX'),

    -- Cohorte Marzo 2024 (6 usuarios)
    (15, '2024-03-01 09:00:00', 'free',    'organic',  'mobile',  'CO'),
    (16, '2024-03-05 12:20:00', 'free',    'social',   'desktop', 'AR'),
    (17, '2024-03-10 17:45:00', 'free',    'ads',      'mobile',  'MX'),
    (18, '2024-03-14 20:30:00', 'free',    'referral', 'mobile',  'PE'),
    (19, '2024-03-20 11:00:00', 'free',    'organic',  'tablet',  'CL'),
    (20, '2024-03-25 14:15:00', 'free',    'ads',      'desktop', 'CO');

-- =============================================
-- SEED: Eventos de MusicFlow (Q1 2024)
-- Journey: app_open → search_song → play_song → create_playlist → subscribe_premium
-- =============================================

-- === ENERO: Usuarios 1-7 ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 1: Journey completo (convierte a premium)
    (1,  1, 'app_open',           '2024-01-03 08:35:00', '{"source":"organic"}'),
    (2,  1, 'search_song',        '2024-01-03 08:40:00', '{"query":"Bad Bunny"}'),
    (3,  1, 'play_song',          '2024-01-03 08:42:00', '{"song":"Monaco","artist":"Bad Bunny"}'),
    (4,  1, 'play_song',          '2024-01-03 08:46:00', '{"song":"Titi Me Pregunto","artist":"Bad Bunny"}'),
    (5,  1, 'create_playlist',    '2024-01-03 09:00:00', '{"name":"Reggaeton Mix"}'),
    (6,  1, 'subscribe_premium',  '2024-01-03 09:15:00', '{"plan":"monthly","price":4.99}'),

    -- Usuario 2: Llega hasta play_song, no crea playlist
    (7,  2, 'app_open',           '2024-01-05 14:20:00', '{"source":"ads"}'),
    (8,  2, 'search_song',        '2024-01-05 14:25:00', '{"query":"Shakira"}'),
    (9,  2, 'play_song',          '2024-01-05 14:28:00', '{"song":"Bzrp Session 53","artist":"Shakira"}'),

    -- Usuario 3: Journey completo (convierte)
    (10, 3, 'app_open',           '2024-01-08 10:05:00', '{"source":"referral"}'),
    (11, 3, 'search_song',        '2024-01-08 10:10:00', '{"query":"Peso Pluma"}'),
    (12, 3, 'play_song',          '2024-01-08 10:12:00', '{"song":"Ella Baila Sola","artist":"Peso Pluma"}'),
    (13, 3, 'create_playlist',    '2024-01-08 10:30:00', '{"name":"Regional Mexicano"}'),
    (14, 3, 'subscribe_premium',  '2024-01-08 10:45:00', '{"plan":"annual","price":49.99}'),

    -- Usuario 4: Solo abre la app y busca, no reproduce
    (15, 4, 'app_open',           '2024-01-12 19:25:00', '{"source":"social"}'),
    (16, 4, 'search_song',        '2024-01-12 19:30:00', '{"query":"Karol G"}'),

    -- Usuario 5: Solo abre la app (bounce)
    (17, 5, 'app_open',           '2024-01-15 11:50:00', '{"source":"organic"}'),

    -- Usuario 6: Llega hasta create_playlist pero no suscribe
    (18, 6, 'app_open',           '2024-01-20 16:35:00', '{"source":"ads"}'),
    (19, 6, 'search_song',        '2024-01-20 16:38:00', '{"query":"Feid"}'),
    (20, 6, 'play_song',          '2024-01-20 16:40:00', '{"song":"Normal","artist":"Feid"}'),
    (21, 6, 'play_song',          '2024-01-20 16:44:00', '{"song":"Ferxxo 100","artist":"Feid"}'),
    (22, 6, 'create_playlist',    '2024-01-20 17:00:00', '{"name":"Feid Hits"}'),

    -- Usuario 7: Llega hasta play_song
    (23, 7, 'app_open',           '2024-01-25 09:05:00', '{"source":"social"}'),
    (24, 7, 'search_song',        '2024-01-25 09:08:00', '{"query":"Rauw Alejandro"}'),
    (25, 7, 'play_song',          '2024-01-25 09:10:00', '{"song":"Todo De Ti","artist":"Rauw Alejandro"}');

-- === FEBRERO: Usuarios 8-14 ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 8: Journey completo (convierte)
    (26, 8, 'app_open',           '2024-02-02 07:50:00', '{"source":"organic"}'),
    (27, 8, 'search_song',        '2024-02-02 07:55:00', '{"query":"J Balvin"}'),
    (28, 8, 'play_song',          '2024-02-02 07:58:00', '{"song":"Mi Gente","artist":"J Balvin"}'),
    (29, 8, 'create_playlist',    '2024-02-02 08:20:00', '{"name":"Perreo"}'),
    (30, 8, 'subscribe_premium',  '2024-02-02 08:30:00', '{"plan":"monthly","price":4.99}'),

    -- Usuario 9: Solo abre y busca
    (31, 9, 'app_open',           '2024-02-06 13:35:00', '{"source":"ads"}'),
    (32, 9, 'search_song',        '2024-02-06 13:40:00', '{"query":"Maluma"}'),

    -- Usuario 10: Llega hasta play_song
    (33, 10, 'app_open',          '2024-02-10 18:05:00', '{"source":"referral"}'),
    (34, 10, 'search_song',       '2024-02-10 18:08:00', '{"query":"Ozuna"}'),
    (35, 10, 'play_song',         '2024-02-10 18:10:00', '{"song":"Se Preparó","artist":"Ozuna"}'),

    -- Usuario 11: Solo abre app (bounce)
    (36, 11, 'app_open',          '2024-02-14 22:15:00', '{"source":"social"}'),

    -- Usuario 12: Llega hasta create_playlist (no suscribe)
    (37, 12, 'app_open',          '2024-02-18 10:20:00', '{"source":"organic"}'),
    (38, 12, 'search_song',       '2024-02-18 10:25:00', '{"query":"Carlos Vives"}'),
    (39, 12, 'play_song',         '2024-02-18 10:28:00', '{"song":"La Bicicleta","artist":"Carlos Vives"}'),
    (40, 12, 'create_playlist',   '2024-02-18 10:50:00', '{"name":"Vallenato Pop"}'),

    -- Usuario 13: Journey completo (convierte)
    (41, 13, 'app_open',          '2024-02-22 15:05:00', '{"source":"ads"}'),
    (42, 13, 'search_song',       '2024-02-22 15:08:00', '{"query":"Anuel AA"}'),
    (43, 13, 'play_song',         '2024-02-22 15:10:00', '{"song":"China","artist":"Anuel AA"}'),
    (44, 13, 'create_playlist',   '2024-02-22 15:30:00', '{"name":"Trap Latino"}'),
    (45, 13, 'subscribe_premium', '2024-02-22 15:45:00', '{"plan":"monthly","price":4.99}'),

    -- Usuario 14: Llega hasta play_song
    (46, 14, 'app_open',          '2024-02-26 08:35:00', '{"source":"referral"}'),
    (47, 14, 'search_song',       '2024-02-26 08:38:00', '{"query":"Natti Natasha"}'),
    (48, 14, 'play_song',         '2024-02-26 08:40:00', '{"song":"Sin Pijama","artist":"Natti Natasha"}');

-- === MARZO: Usuarios 15-20 ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 15: Journey completo (convierte)
    (49, 15, 'app_open',          '2024-03-01 09:05:00', '{"source":"organic"}'),
    (50, 15, 'search_song',       '2024-03-01 09:10:00', '{"query":"Mora"}'),
    (51, 15, 'play_song',         '2024-03-01 09:12:00', '{"song":"Una Vez","artist":"Mora"}'),
    (52, 15, 'create_playlist',   '2024-03-01 09:30:00', '{"name":"Chill Reggaeton"}'),
    (53, 15, 'subscribe_premium', '2024-03-01 09:50:00', '{"plan":"annual","price":49.99}'),

    -- Usuario 16: Solo abre y busca
    (54, 16, 'app_open',          '2024-03-05 12:25:00', '{"source":"social"}'),
    (55, 16, 'search_song',       '2024-03-05 12:30:00', '{"query":"Duki"}'),

    -- Usuario 17: Llega hasta play_song
    (56, 17, 'app_open',          '2024-03-10 17:50:00', '{"source":"ads"}'),
    (57, 17, 'search_song',       '2024-03-10 17:53:00', '{"query":"Myke Towers"}'),
    (58, 17, 'play_song',         '2024-03-10 17:55:00', '{"song":"La Nota","artist":"Myke Towers"}'),

    -- Usuario 18: Llega hasta create_playlist (no suscribe)
    (59, 18, 'app_open',          '2024-03-14 20:35:00', '{"source":"referral"}'),
    (60, 18, 'search_song',       '2024-03-14 20:38:00', '{"query":"Sebastian Yatra"}'),
    (61, 18, 'play_song',         '2024-03-14 20:40:00', '{"song":"Traicionera","artist":"Sebastian Yatra"}'),
    (62, 18, 'create_playlist',   '2024-03-14 21:00:00', '{"name":"Pop Latino"}'),

    -- Usuario 19: Solo abre app (bounce)
    (63, 19, 'app_open',          '2024-03-20 11:05:00', '{"source":"organic"}'),

    -- Usuario 20: Llega hasta play_song
    (64, 20, 'app_open',          '2024-03-25 14:20:00', '{"source":"ads"}'),
    (65, 20, 'search_song',       '2024-03-25 14:23:00', '{"query":"Ryan Castro"}'),
    (66, 20, 'play_song',         '2024-03-25 14:25:00', '{"song":"Muevelo","artist":"Ryan Castro"}');

-- === EVENTOS DE RETORNO (usuarios que regresan después) ===
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
    -- Usuario 1 regresa semana 1 y semana 3
    (67, 1, 'app_open',    '2024-01-10 20:00:00', '{"source":"direct"}'),
    (68, 1, 'play_song',   '2024-01-10 20:05:00', '{"song":"Yonaguni","artist":"Bad Bunny"}'),
    (69, 1, 'app_open',    '2024-01-24 18:00:00', '{"source":"notification"}'),
    (70, 1, 'play_song',   '2024-01-24 18:10:00', '{"song":"Dakiti","artist":"Bad Bunny"}'),

    -- Usuario 3 regresa semana 2
    (71, 3, 'app_open',    '2024-01-20 14:00:00', '{"source":"notification"}'),
    (72, 3, 'play_song',   '2024-01-20 14:05:00', '{"song":"Lady Gaga","artist":"Peso Pluma"}'),

    -- Usuario 6 regresa semana 1
    (73, 6, 'app_open',    '2024-01-27 10:00:00', '{"source":"organic"}'),
    (74, 6, 'play_song',   '2024-01-27 10:05:00', '{"song":"Feliz Cumpleaños","artist":"Feid"}'),

    -- Usuario 8 regresa semana 1 y semana 2
    (75, 8, 'app_open',    '2024-02-09 08:00:00', '{"source":"notification"}'),
    (76, 8, 'play_song',   '2024-02-09 08:10:00', '{"song":"Colores","artist":"J Balvin"}'),
    (77, 8, 'app_open',    '2024-02-17 19:00:00', '{"source":"direct"}'),
    (78, 8, 'play_song',   '2024-02-17 19:05:00', '{"song":"Agua","artist":"J Balvin"}'),

    -- Usuario 13 regresa semana 1
    (79, 13, 'app_open',   '2024-02-29 16:00:00', '{"source":"organic"}'),
    (80, 13, 'play_song',  '2024-02-29 16:10:00', '{"song":"Secreto","artist":"Anuel AA"}'),

    -- Usuario 15 regresa semana 1 y semana 2
    (81, 15, 'app_open',   '2024-03-08 12:00:00', '{"source":"notification"}'),
    (82, 15, 'play_song',  '2024-03-08 12:05:00', '{"song":"Memorias","artist":"Mora"}'),
    (83, 15, 'app_open',   '2024-03-16 09:00:00', '{"source":"direct"}'),

    -- Usuario 2 regresa semana 2 (tardío)
    (84, 2, 'app_open',    '2024-01-19 11:00:00', '{"source":"ads"}'),
    (85, 2, 'play_song',   '2024-01-19 11:05:00', '{"song":"Waka Waka","artist":"Shakira"}'),

    -- Usuario 10 regresa semana 1
    (86, 10, 'app_open',   '2024-02-17 10:00:00', '{"source":"referral"}'),
    (87, 10, 'play_song',  '2024-02-17 10:05:00', '{"song":"Dile Que Tu Me Quieres","artist":"Ozuna"}');
```

### 0.3 Verificación rápida

Ejecuta estas consultas para confirmar que todo cargó correctamente:

```sql
-- ¿Cuántos usuarios?
SELECT COUNT(*) AS total_users FROM users;
-- Esperado: 20

-- ¿Cuántos eventos?
SELECT COUNT(*) AS total_events FROM events;
-- Esperado: 87

-- Distribución de eventos por tipo
SELECT event_name, COUNT(*) AS cantidad
FROM events
GROUP BY event_name
ORDER BY cantidad DESC;
```

---
# Bloque 0 · Breakout Rooms — Calentamiento (10 min)

**Instrucciones:** Trabaja en parejas o triadas. Resuelvan las 3 tareas y comparen respuestas antes de volver a la plenaria.

---

### Ejercicio 0.1: Exploración de eventos únicos (Nivel: ⭐)

**Descripción:** Lista todos los tipos de evento (`event_name`) distintos que existen en la tabla `events`, ordenados alfabéticamente.

**Objetivo:** Familiarizarte con los datos y confirmar que los 5 tipos de evento del journey de MusicFlow están presentes.

**Explicación:** Antes de construir cualquier funnel, necesitamos saber exactamente qué eventos tenemos disponibles. `DISTINCT` elimina duplicados y nos da una vista limpia.

**Pista:** Usa `SELECT DISTINCT event_name FROM events ORDER BY 1`.

---

**✅ Solución:**
```sql
SELECT DISTINCT event_name
FROM events
ORDER BY 1;
```

> **Resultado esperado:** 5 eventos: `app_open`, `create_playlist`, `play_song`, `search_song`, `subscribe_premium`.

### Ejercicio 0.2: Usuarios activos por mes (Nivel: ⭐)

**Descripción:** Cuenta cuántos **usuarios únicos** abrieron la app (`app_open`) en cada mes del Q1 2024.

**Objetivo:** Practicar `COUNT(DISTINCT ...)` con agrupación temporal para entender la distribución de actividad.

**Explicación:** `DATE_TRUNC('month', event_ts)` trunca cualquier timestamp al primer día de su mes. Al agrupar por este valor, obtenemos conteos mensuales limpios.

**Pista:** Filtra por `event_name = 'app_open'`, usa `DATE_TRUNC('month', event_ts)` como columna de agrupación, y cuenta `DISTINCT user_id`.

---

**✅ Solución:**
```sql
SELECT
    DATE_TRUNC('month', event_ts) AS mes,
    COUNT(DISTINCT user_id) AS usuarios_unicos
FROM events
WHERE event_name = 'app_open'
GROUP BY mes
ORDER BY mes;
```

> **Resultado esperado:** Enero ~7, Febrero ~8 (incluye retornos), Marzo ~7 (incluye retornos).

### Ejercicio 0.3: Usuarios con playlist pero sin suscripción (Nivel: ⭐⭐)

**Descripción:** Identifica los usuarios que crearon una playlist (`create_playlist`) pero **nunca** se suscribieron a premium (`subscribe_premium`).

**Objetivo:** Detectar journeys incompletos usando la técnica de "set difference" con CTEs y LEFT JOIN.

**Explicación:** Creamos dos CTEs: una con los usuarios que hicieron `create_playlist` y otra con los que hicieron `subscribe_premium`. Luego hacemos un LEFT JOIN y filtramos donde la segunda sea NULL — esos son los usuarios que crearon playlist pero no convirtieron.

**Pista:** Crea dos CTEs con `SELECT DISTINCT user_id`, una para cada evento. Haz `LEFT JOIN` entre la CTE de playlist y la de subscribe. Filtra `WHERE subscribe_cte.user_id IS NULL`.

---

**✅ Solución:**
```sql
WITH playlist_users AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'create_playlist'
),
premium_users AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'subscribe_premium'
)
SELECT p.user_id
FROM playlist_users p
LEFT JOIN premium_users s ON p.user_id = s.user_id
WHERE s.user_id IS NULL
ORDER BY p.user_id;
```

> **Resultado esperado:** Usuarios 6, 12 y 18 — crearon playlist pero no se suscribieron. Estos son targets ideales para una campaña de conversión.

---
# Bloque 1 · Coding Together — CTEs aplicadas (25 min)

Vamos a practicar CTEs con ejercicios progresivos aplicados al escenario de MusicFlow.

---

### Ejercicio 1.1: Resumen de actividad por usuario con CTE (Nivel: ⭐)

**Descripción:** Crea una CTE llamada `actividad` que calcule para cada usuario: su cantidad total de eventos y la fecha de su primer y último evento. En la consulta final, muestra solo los 5 usuarios más activos.

**Objetivo:** Practicar la sintaxis básica de CTEs con funciones de agregación.

**Explicación:** Una CTE nos permite separar el cálculo del resumen (paso 1) de la presentación filtrada (paso 2). Sin CTEs, tendríamos que anidar el GROUP BY dentro de un subquery, lo cual es menos legible.

**Pista:** Dentro de la CTE, usa `COUNT(*)`, `MIN(event_ts)` y `MAX(event_ts)` agrupando por `user_id`. En el SELECT final, ordena por `total_eventos DESC` y aplica `LIMIT 5`.

---

**✅ Solución:**
```sql
WITH actividad AS (
    SELECT
        user_id,
        COUNT(*)        AS total_eventos,
        MIN(event_ts)   AS primer_evento,
        MAX(event_ts)   AS ultimo_evento
    FROM events
    GROUP BY user_id
)
SELECT *
FROM actividad
ORDER BY total_eventos DESC
LIMIT 5;
```

> **Resultado esperado:** El usuario 1 lidera con ~10 eventos (journey completo + retornos), seguido del usuario 8 con ~8-9.

### Ejercicio 1.2: Dos CTEs con JOIN — Usuarios premium y su país (Nivel: ⭐)

**Descripción:** Crea una CTE `suscriptores` con los usuarios que hicieron `subscribe_premium` y otra CTE `info_user` con datos de la tabla `users` (país y canal). Crúzalas para mostrar qué suscriptores premium vienen de qué país y canal.

**Objetivo:** Practicar múltiples CTEs combinadas con JOIN para enriquecer datos.

**Explicación:** Cuando necesitas combinar información de diferentes fuentes (eventos + atributos del usuario), definir cada fuente como una CTE hace que el JOIN final sea limpio y fácil de entender.

**Pista:** La primera CTE filtra `events` por `subscribe_premium` con `DISTINCT user_id`. La segunda selecciona `user_id, country, channel` de `users`. El SELECT final cruza ambas con `INNER JOIN`.

---

**✅ Solución:**
```sql
WITH suscriptores AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'subscribe_premium'
),
info_user AS (
    SELECT user_id, country, channel, device
    FROM users
)
SELECT
    i.user_id,
    i.country,
    i.channel,
    i.device
FROM suscriptores s
INNER JOIN info_user i ON s.user_id = i.user_id
ORDER BY i.country, i.user_id;
```

> **Resultado esperado:** 5 suscriptores premium — los usuarios 1, 3, 8, 13 y 15, con sus países (CO, AR, CO, CL, CO) y canales.

### Ejercicio 1.3: CTEs encadenadas — Pipeline de transformaciones (Nivel: ⭐⭐)

**Descripción:** Construye 3 CTEs encadenadas:
1. `eventos_febrero`: filtra solo eventos de febrero 2024.
2. `conteo_por_usuario`: cuenta eventos por usuario en febrero.
3. `segmento`: clasifica usuarios en "Alta actividad" (5+ eventos), "Media" (3-4) o "Baja" (1-2).

En la consulta final, muestra cuántos usuarios hay en cada segmento.

**Objetivo:** Demostrar que cada CTE puede referenciar la anterior, construyendo un pipeline de transformaciones.

**Explicación:** Las CTEs se evalúan en orden. `conteo_por_usuario` referencia a `eventos_febrero` (no a la tabla completa). `segmento` referencia a `conteo_por_usuario`. Esto es como una cadena de producción donde cada paso refina el anterior.

**Pista:** En la CTE `segmento`, usa `CASE WHEN total_eventos >= 5 THEN 'Alta actividad' WHEN total_eventos >= 3 THEN 'Media' ELSE 'Baja' END`.

---

**✅ Solución:**
```sql
WITH eventos_febrero AS (
    SELECT *
    FROM events
    WHERE event_ts >= '2024-02-01'
      AND event_ts <  '2024-03-01'
),
conteo_por_usuario AS (
    SELECT
        user_id,
        COUNT(*) AS total_eventos
    FROM eventos_febrero
    GROUP BY user_id
),
segmento AS (
    SELECT
        user_id,
        total_eventos,
        CASE
            WHEN total_eventos >= 5 THEN 'Alta actividad'
            WHEN total_eventos >= 3 THEN 'Media actividad'
            ELSE 'Baja actividad'
        END AS nivel
    FROM conteo_por_usuario
)
SELECT
    nivel,
    COUNT(*) AS num_usuarios,
    ROUND(AVG(total_eventos), 1) AS promedio_eventos
FROM segmento
GROUP BY nivel
ORDER BY promedio_eventos DESC;
```

> **Resultado esperado:** Tres segmentos con distribución variada. Los usuarios con journey completo + retorno estarán en "Alta actividad".

### Ejercicio 1.4: CTE con Window Function — Primer evento de cada usuario (Nivel: ⭐⭐⭐)

**Descripción:** Usa una CTE con `ROW_NUMBER()` para encontrar el **primer evento** de cada usuario. Luego cruza con `users` para mostrar: user_id, country, channel, primer_evento (nombre) y su timestamp.

**Objetivo:** Combinar CTEs con funciones de ventana y JOINs — técnica esencial para análisis de journey.

**Explicación:** `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts)` asigna un número secuencial a cada evento de cada usuario, ordenado cronológicamente. Filtrando `WHERE rn = 1` obtenemos exactamente el primer evento.

**Pista:** Crea la CTE con `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts) AS rn`. Filtra `rn = 1` en el JOIN final con `users`.

---

**✅ Solución:**
```sql
WITH primer_evento AS (
    SELECT
        user_id,
        event_name,
        event_ts,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts) AS rn
    FROM events
)
SELECT
    u.user_id,
    u.country,
    u.channel,
    pe.event_name   AS primer_evento,
    pe.event_ts     AS fecha_primer_evento
FROM users u
INNER JOIN primer_evento pe
    ON u.user_id = pe.user_id
   AND pe.rn = 1
ORDER BY u.user_id;
```

> **Resultado esperado:** Los 20 usuarios, todos con `app_open` como primer evento (es lo esperable en el journey de MusicFlow).

---
# Bloque 2 · Coding Together — Funnel de conversión (25 min)

Ahora aplicamos CTEs para construir el funnel de MusicFlow:
```
app_open → search_song → play_song → create_playlist → subscribe_premium
```

---

### Ejercicio 2.1: Funnel completo Q1 2024 (Nivel: ⭐⭐)

**Descripción:** Construye el funnel completo del Q1 2024 (enero–marzo). Usa una CTE por etapa y muestra: conteo de usuarios por etapa, drop-off entre etapas consecutivas, y tasa de conversión global (app_open → subscribe_premium).

**Objetivo:** Construir un funnel completo de 5 etapas con CTEs, calcular drop-off y conversión.

**Explicación:** Cada CTE selecciona los `user_id` distintos que alcanzaron esa etapa. La CTE `counts` consolida los 5 conteos. El SELECT final calcula las métricas de negocio: drop-off (resta entre etapas) y conversión global (porcentaje de usuarios que van de inicio a fin).

**Pista:** Crea 5 CTEs (`f_open`, `f_search`, `f_play`, `f_playlist`, `f_premium`) con `DISTINCT user_id`. Usa una CTE `counts` con subconsultas escalares `(SELECT COUNT(*) FROM ...)`. Calcula drop-off como resta y conversión como `100.0 * premium / open`.

---

**✅ Solución:**
```sql
WITH f_open AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'app_open'
),
f_search AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'search_song'
),
f_play AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'play_song'
),
f_playlist AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'create_playlist'
),
f_premium AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'subscribe_premium'
),
counts AS (
    SELECT
        (SELECT COUNT(*) FROM f_open)     AS n_open,
        (SELECT COUNT(*) FROM f_search)   AS n_search,
        (SELECT COUNT(*) FROM f_play)     AS n_play,
        (SELECT COUNT(*) FROM f_playlist) AS n_playlist,
        (SELECT COUNT(*) FROM f_premium)  AS n_premium
)
SELECT
    n_open,
    n_search,
    n_play,
    n_playlist,
    n_premium,
    -- Drop-off entre etapas
    (n_open     - n_search)   AS drop_open_to_search,
    (n_search   - n_play)     AS drop_search_to_play,
    (n_play     - n_playlist) AS drop_play_to_playlist,
    (n_playlist - n_premium)  AS drop_playlist_to_premium,
    -- Conversión global
    ROUND(100.0 * n_premium / NULLIF(n_open, 0), 2) AS conversion_rate_pct
FROM counts;
```

> **Resultado esperado:** 20 → 17 → 15 → 8 → 5. La mayor caída es de `play_song` a `create_playlist` (7 usuarios perdidos). Conversión global: 25%.

### Ejercicio 2.2: Funnel solo de febrero 2024 (Nivel: ⭐⭐)

**Descripción:** Construye el funnel filtrando solo los eventos de **febrero 2024**. Compara los resultados con el funnel general para detectar diferencias.

**Objetivo:** Practicar filtrado temporal dentro de CTEs y comparar funnels entre periodos.

**Explicación:** Agrega una CTE `base` que filtre eventos con `WHERE event_ts >= '2024-02-01' AND event_ts < '2024-03-01'`. Todas las CTEs de etapa deben usar `base` en lugar de `events`. Esto garantiza que el funnel solo refleje actividad de febrero.

**Pista:** Idéntico al ejercicio anterior, pero con una CTE `base` al inicio que filtra por rango de fechas. Las demás CTEs hacen `FROM base WHERE event_name = '...'`.

---

**✅ Solución:**
```sql
WITH base AS (
    SELECT *
    FROM events
    WHERE event_ts >= '2024-02-01'
      AND event_ts <  '2024-03-01'
),
f_open AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'app_open'
),
f_search AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'search_song'
),
f_play AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'play_song'
),
f_playlist AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'create_playlist'
),
f_premium AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'subscribe_premium'
),
counts AS (
    SELECT
        (SELECT COUNT(*) FROM f_open)     AS n_open,
        (SELECT COUNT(*) FROM f_search)   AS n_search,
        (SELECT COUNT(*) FROM f_play)     AS n_play,
        (SELECT COUNT(*) FROM f_playlist) AS n_playlist,
        (SELECT COUNT(*) FROM f_premium)  AS n_premium
)
SELECT
    'Febrero 2024' AS periodo,
    n_open, n_search, n_play, n_playlist, n_premium,
    (n_open   - n_search)   AS drop_open_to_search,
    (n_search - n_play)     AS drop_search_to_play,
    (n_play   - n_playlist) AS drop_play_to_playlist,
    (n_playlist - n_premium) AS drop_playlist_to_premium,
    ROUND(100.0 * n_premium / NULLIF(n_open, 0), 2) AS conversion_rate_pct
FROM counts;
```

> **Resultado esperado:** En febrero hay ~9 usuarios que abren (7 nuevos + retornos), y la conversión sigue un patrón similar al general. Observa si hay diferencias en los puntos de caída.

### Ejercicio 2.3: Tiempos entre etapas del journey (Nivel: ⭐⭐⭐)

**Descripción:** Para cada usuario, calcula el **tiempo en minutos** entre cada par de etapas consecutivas del journey (open→search, search→play, play→playlist, playlist→premium). Usa el primer timestamp de cada evento por usuario.

**Objetivo:** Medir latencias entre etapas para identificar dónde los usuarios tardan más (posible fricción en la UX).

**Explicación:** Creamos una CTE por etapa con el `MIN(event_ts)` de ese evento por usuario (su primer ocurrencia). Luego hacemos LEFT JOINs para reunir todos los timestamps en una fila por usuario. La diferencia entre timestamps nos da los minutos entre etapas. `NULL` indica que el usuario no llegó a esa etapa.

**Pista:** Usa `DATEDIFF('minute', ts_anterior, ts_actual)` en DuckDB para calcular la diferencia en minutos. Crea CTEs `o`, `s`, `p`, `pl`, `pr` con `MIN(event_ts) AS ts` agrupando por `user_id`.

---

**✅ Solución:**
```sql
WITH
o  AS (SELECT user_id, MIN(event_ts) AS ts FROM events WHERE event_name='app_open'          GROUP BY user_id),
s  AS (SELECT user_id, MIN(event_ts) AS ts FROM events WHERE event_name='search_song'       GROUP BY user_id),
p  AS (SELECT user_id, MIN(event_ts) AS ts FROM events WHERE event_name='play_song'         GROUP BY user_id),
pl AS (SELECT user_id, MIN(event_ts) AS ts FROM events WHERE event_name='create_playlist'   GROUP BY user_id),
pr AS (SELECT user_id, MIN(event_ts) AS ts FROM events WHERE event_name='subscribe_premium' GROUP BY user_id)
SELECT
    o.user_id,
    DATEDIFF('minute', o.ts,  s.ts)  AS min_open_to_search,
    DATEDIFF('minute', s.ts,  p.ts)  AS min_search_to_play,
    DATEDIFF('minute', p.ts,  pl.ts) AS min_play_to_playlist,
    DATEDIFF('minute', pl.ts, pr.ts) AS min_playlist_to_premium
FROM o
LEFT JOIN s  ON o.user_id = s.user_id
LEFT JOIN p  ON o.user_id = p.user_id
LEFT JOIN pl ON o.user_id = pl.user_id
LEFT JOIN pr ON o.user_id = pr.user_id
ORDER BY o.user_id;
```

> **Resultado esperado:** Los usuarios que convirtieron muestran tiempos de ~5-45 minutos entre etapas. `NULL` indica que el usuario no alcanzó esa etapa. El paso más lento suele ser `play_song → create_playlist` (mayor fricción).

### 2.4 Discusión: Interpretación del funnel

Después de revisar los resultados, discute con tu equipo:

1. **¿Cuál es el mayor cuello de botella?** (la etapa con mayor drop-off)
2. **¿Qué hipótesis podrías formular?** Por ejemplo:
   - "Los usuarios que escuchan música pero no crean playlists necesitan un *nudge* en la UI."
   - "El paso de playlist a premium podría mejorarse con un trial gratuito."
3. **¿Qué acción concreta propondrías** al equipo de producto de MusicFlow?

> Este es el tipo de análisis que un Data Analyst presenta al equipo: datos + hipótesis + recomendación accionable.

---
# Bloque 3 · Coding Together — Cohorts y retención (20 min)

---

### Ejercicio 3.1: Identificar cohorts mensuales (Nivel: ⭐)

**Descripción:** Muestra las cohortes mensuales de MusicFlow: para cada mes de signup, cuenta cuántos usuarios hay y cuántos de ellos eventualmente se suscribieron a premium.

**Objetivo:** Asignar cohorts y calcular una métrica simple por cohort.

**Explicación:** `DATE_TRUNC('month', signup_ts)` agrupa a cada usuario por su mes de registro. Al cruzar con la tabla `events`, podemos saber cuántos de cada cohort llegaron a `subscribe_premium`.

**Pista:** Usa un LEFT JOIN entre `users` y una subconsulta/CTE que tenga los `user_id` con `subscribe_premium`. Agrupa por el mes truncado.

---

**✅ Solución:**
```sql
WITH premium AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'subscribe_premium'
)
SELECT
    DATE_TRUNC('month', u.signup_ts)  AS cohort_month,
    COUNT(*)                          AS cohort_size,
    COUNT(p.user_id)                  AS premium_users,
    ROUND(100.0 * COUNT(p.user_id) / COUNT(*), 2) AS premium_rate_pct
FROM users u
LEFT JOIN premium p ON u.user_id = p.user_id
GROUP BY cohort_month
ORDER BY cohort_month;
```

> **Resultado esperado:** Enero: 7 usuarios (2 premium, ~28.6%), Febrero: 7 usuarios (2 premium, ~28.6%), Marzo: 6 usuarios (1 premium, ~16.7%).

### Ejercicio 3.2: Tabla de retención semanal (Nivel: ⭐⭐⭐)

**Descripción:** Construye una tabla de retención que muestre, para cada cohort mensual, el porcentaje de usuarios que tuvieron al menos un evento en las semanas 0, 1, 2 y 3 después de su mes de registro.

**Objetivo:** Construir la tabla de retención completa usando CTEs, CROSS JOIN y lógica de ventanas temporales.

**Explicación:**
- Paso 1: Asignar cada usuario a su cohort mensual.
- Paso 2: Generar un grid de semanas (0, 1, 2, 3) con `UNNEST`.
- Paso 3: Para cada combinación (cohort, semana), contar cuántos usuarios tuvieron al menos un evento en el rango `[7*w, 7*(w+1))` días desde el inicio del cohort.
- Paso 4: Calcular el porcentaje de retención.

**Pista:** La condición de retención es: `DATEDIFF('day', cohort_month, CAST(event_ts AS DATE))` debe estar entre `7*w` y `7*(w+1)`. Usa `CASE WHEN EXISTS (...)` para verificar si cada usuario tiene un evento en esa ventana.

---

**✅ Solución:**
```sql
WITH
u AS (
    SELECT user_id, DATE_TRUNC('month', signup_ts) AS cohort_month
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),
retention AS (
    SELECT
        u.cohort_month,
        g.week_offset,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1 FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN u.user_id
        END) AS retained_users,
        COUNT(DISTINCT u.user_id) AS cohort_users
    FROM u
    CROSS JOIN grid g
    GROUP BY u.cohort_month, g.week_offset
)
SELECT
    cohort_month,
    week_offset,
    retained_users,
    cohort_users,
    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
FROM retention
ORDER BY cohort_month, week_offset;
```

> **Resultado esperado:** La semana 0 tiene retención alta (usuarios activos al registrarse), cayendo en semanas posteriores. Los cohorts con más suscriptores premium tienden a tener mejor retención.

### Ejercicio 3.3: Tabla pivote para heatmap (Nivel: ⭐⭐⭐)

**Descripción:** Transforma la tabla de retención del ejercicio anterior en formato **ancho** (pivote): una fila por cohort, una columna por semana (w0, w1, w2, w3) con el porcentaje de retención.

**Objetivo:** Practicar pivote manual con `MAX(CASE WHEN ...)` para generar el formato ideal para un heatmap en Google Sheets.

**Explicación:** SQL no tiene PIVOT nativo en todos los motores. Simulamos el pivote con `MAX(CASE WHEN week_offset = N THEN valor END)` dentro de un `GROUP BY cohort_month`. Cada columna resultante contiene el porcentaje para esa semana específica.

**Pista:** Toma la CTE `retention` del ejercicio anterior. Agrega una CTE `pct` que calcule el porcentaje. En el SELECT final, usa `MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0`, etc.

---

**✅ Solución:**
```sql
WITH
u AS (
    SELECT user_id, DATE_TRUNC('month', signup_ts) AS cohort_month
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),
retention AS (
    SELECT
        u.cohort_month,
        g.week_offset,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1 FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN u.user_id
        END) AS retained_users,
        COUNT(DISTINCT u.user_id) AS cohort_users
    FROM u CROSS JOIN grid g
    GROUP BY u.cohort_month, g.week_offset
),
pct AS (
    SELECT
        cohort_month,
        week_offset,
        ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
    FROM retention
)
SELECT
    cohort_month,
    MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0,
    MAX(CASE WHEN week_offset = 1 THEN retention_pct END) AS w1,
    MAX(CASE WHEN week_offset = 2 THEN retention_pct END) AS w2,
    MAX(CASE WHEN week_offset = 3 THEN retention_pct END) AS w3
FROM pct
GROUP BY cohort_month
ORDER BY cohort_month;
```

> **Resultado esperado:** Tabla de 3 filas (ene, feb, mar) × 4 columnas (w0–w3). Copia este resultado a Google Sheets y aplica Formato condicional → Escala de color para obtener un heatmap visual.

### Ejercicio 3.4: Retención segmentada por canal (Nivel: ⭐⭐⭐)

**Descripción:** Extiende la tabla de retención para segmentarla por **canal de adquisición** (`channel`). Muestra la retención por (cohort_month, channel, week_offset).

**Objetivo:** Comparar la retención entre segmentos de negocio para responder: "¿Los usuarios de referral retienen mejor que los de ads?"

**Explicación:** La única modificación respecto al ejercicio anterior es incluir `u.channel` en la CTE `u`, agregarlo a todos los `GROUP BY`, y mostrarlo en el SELECT final. Esto nos permite comparar curvas de retención por canal.

**Pista:** Agrega `channel` a la CTE `u` (trayéndola de `users`). Agrega `u.channel` a los `GROUP BY` de `retention`. Ordena por `cohort_month, channel, week_offset`.

---

**✅ Solución:**
```sql
WITH
u AS (
    SELECT user_id, DATE_TRUNC('month', signup_ts) AS cohort_month, channel
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),
retention AS (
    SELECT
        u.cohort_month,
        u.channel,
        g.week_offset,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1 FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN u.user_id
        END) AS retained_users,
        COUNT(DISTINCT u.user_id) AS cohort_users
    FROM u CROSS JOIN grid g
    GROUP BY u.cohort_month, u.channel, g.week_offset
)
SELECT
    cohort_month,
    channel,
    week_offset,
    retained_users,
    cohort_users,
    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
FROM retention
ORDER BY cohort_month, channel, week_offset;
```

> **Resultado esperado:** Tabla segmentada por canal. Observa: ¿los usuarios de `referral` muestran mejor retención que los de `ads`? Esto es clave para decidir dónde invertir el presupuesto de adquisición.

---
# Bloque 4 · Breakout Rooms — Ejercicio integrador (15 min)

**Instrucciones:** Trabaja en equipo (2-3 personas). Resuelvan los ejercicios y prepárense para discutir los hallazgos en plenaria.

---

### Ejercicio 4.1: Journey individual con tiempos (Nivel: ⭐⭐)

**Descripción:** Reconstruye el journey completo del **usuario 8**: todos sus eventos ordenados cronológicamente, con el tiempo transcurrido (en minutos) entre cada evento y el anterior.

**Objetivo:** Usar `LAG()` para medir tiempos entre pasos del journey de un usuario real.

**Explicación:** `LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts)` nos da el timestamp del evento inmediatamente anterior del mismo usuario. La diferencia con el evento actual nos dice cuántos minutos tardó en pasar al siguiente paso.

**Pista:** Crea una CTE `journey` con `LAG(event_ts) OVER (...)`. En el SELECT final, usa `DATEDIFF('minute', prev_ts, event_ts)`.

---

**✅ Solución:**
```sql
WITH journey AS (
    SELECT
        user_id,
        event_name,
        event_ts,
        LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) AS prev_ts
    FROM events
    WHERE user_id = 8
)
SELECT
    user_id,
    event_name,
    event_ts,
    prev_ts,
    DATEDIFF('minute', prev_ts, event_ts) AS minutos_desde_anterior
FROM journey
ORDER BY event_ts;
```

> **Resultado esperado:** El usuario 8 tiene su journey inicial (open→search→play→playlist→premium en ~40 min) y luego regresa en semana 1 y 2. Los tiempos de retorno son de miles de minutos (días).

### Ejercicio 4.2: Funnel por país (Nivel: ⭐⭐)

**Descripción:** Construye un funnel resumido (`app_open` → `subscribe_premium`) agrupado por **país**. Muestra: país, usuarios que abren, usuarios premium, y tasa de conversión.

**Objetivo:** Segmentar el funnel por un atributo de usuario para identificar qué mercados convierten mejor.

**Explicación:** Haz un JOIN entre `events` y `users` para obtener el país de cada usuario. Luego usa `CASE WHEN` dentro de `COUNT(DISTINCT ...)` para contar usuarios por etapa, agrupados por país.

**Pista:** Usa una CTE `base` con el JOIN events+users. En el SELECT final, agrupa por `country` y usa `COUNT(DISTINCT CASE WHEN event_name = '...' THEN user_id END)`.

---

**✅ Solución:**
```sql
WITH base AS (
    SELECT e.user_id, e.event_name, u.country
    FROM events e
    INNER JOIN users u ON e.user_id = u.user_id
)
SELECT
    country,
    COUNT(DISTINCT CASE WHEN event_name = 'app_open'          THEN user_id END) AS open_users,
    COUNT(DISTINCT CASE WHEN event_name = 'search_song'       THEN user_id END) AS search_users,
    COUNT(DISTINCT CASE WHEN event_name = 'play_song'         THEN user_id END) AS play_users,
    COUNT(DISTINCT CASE WHEN event_name = 'subscribe_premium' THEN user_id END) AS premium_users,
    ROUND(100.0 *
        COUNT(DISTINCT CASE WHEN event_name = 'subscribe_premium' THEN user_id END) /
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'app_open' THEN user_id END), 0),
    2) AS conversion_rate_pct
FROM base
GROUP BY country
ORDER BY conversion_rate_pct DESC;
```

> **Resultado esperado:** CO tiene la mayor cantidad de usuarios pero observa si su tasa de conversión es mayor o menor que países más pequeños como CL o PE.

### Ejercicio 4.3: Clasificación de usuarios por avance en el journey (Nivel: ⭐⭐⭐)

**Descripción:** Clasifica cada usuario según la **etapa máxima** que alcanzó en su journey:
- `'Bounce'`: solo hizo `app_open`
- `'Explorador'`: llegó hasta `search_song`
- `'Oyente'`: llegó hasta `play_song`
- `'Creador'`: llegó hasta `create_playlist`
- `'Premium'`: completó `subscribe_premium`

Muestra cuántos usuarios hay en cada categoría y su porcentaje del total.

**Objetivo:** Construir una clasificación avanzada combinando `MAX(CASE WHEN ...)` con lógica condicional para segmentar la base de usuarios.

**Explicación:** Primero, para cada usuario determinamos qué etapas completó usando flags booleanos (`MAX(CASE WHEN event_name = '...' THEN 1 ELSE 0 END)`). Luego, con `CASE WHEN` en cascada (de mayor a menor avance), asignamos la categoría. Esto es como recorrer el embudo de abajo hacia arriba.

**Pista:** Crea una CTE `etapas` con flags por usuario. Crea una CTE `clasificacion` con `CASE WHEN tiene_premium = 1 THEN 'Premium' WHEN tiene_playlist = 1 THEN 'Creador' ...`. Agrupa y cuenta en el SELECT final.

---

**✅ Solución:**
```sql
WITH etapas AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'app_open'          THEN 1 ELSE 0 END) AS tiene_open,
        MAX(CASE WHEN event_name = 'search_song'       THEN 1 ELSE 0 END) AS tiene_search,
        MAX(CASE WHEN event_name = 'play_song'         THEN 1 ELSE 0 END) AS tiene_play,
        MAX(CASE WHEN event_name = 'create_playlist'   THEN 1 ELSE 0 END) AS tiene_playlist,
        MAX(CASE WHEN event_name = 'subscribe_premium' THEN 1 ELSE 0 END) AS tiene_premium
    FROM events
    GROUP BY user_id
),
clasificacion AS (
    SELECT
        user_id,
        CASE
            WHEN tiene_premium  = 1 THEN 'Premium'
            WHEN tiene_playlist = 1 THEN 'Creador'
            WHEN tiene_play     = 1 THEN 'Oyente'
            WHEN tiene_search   = 1 THEN 'Explorador'
            WHEN tiene_open     = 1 THEN 'Bounce'
            ELSE 'Sin actividad'
        END AS categoria
    FROM etapas
)
SELECT
    categoria,
    COUNT(*) AS num_usuarios,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clasificacion), 2) AS porcentaje
FROM clasificacion
GROUP BY categoria
ORDER BY
    CASE categoria
        WHEN 'Bounce'     THEN 1
        WHEN 'Explorador' THEN 2
        WHEN 'Oyente'     THEN 3
        WHEN 'Creador'    THEN 4
        WHEN 'Premium'    THEN 5
    END;
```

> **Resultado esperado:** Bounce: 3 (15%), Explorador: 3 (15%), Oyente: 6 (30%), Creador: 3 (15%), Premium: 5 (25%). La mayor concentración está en "Oyente" — usuarios que escuchan pero no se comprometen más. Oportunidad clara de conversión.

### Ejercicio 4.4: Checks de calidad de datos (Nivel: ⭐⭐)

**Descripción:** Antes de presentar cualquier análisis, el Data Analyst debe validar la calidad de los datos. Ejecuta los siguientes checks y documenta tus hallazgos:

1. **Duplicados exactos:** ¿Hay combinaciones `(user_id, event_name, event_ts)` repetidas?
2. **Eventos en el futuro:** ¿Hay eventos con timestamp posterior a la fecha actual?
3. **Journeys saltados:** ¿Hay usuarios que hicieron `subscribe_premium` sin haber hecho `play_song`?

**Objetivo:** Practicar validaciones de calidad como paso obligatorio antes de análisis.

**Explicación:** Estos checks son la "higiene" del análisis de datos. Un funnel construido sobre datos con duplicados da resultados inflados. Eventos en el futuro indican errores de logging. Journeys saltados pueden indicar bugs o flujos alternativos del producto.

**Pista:** Para duplicados, usa `GROUP BY ... HAVING COUNT(*) > 1`. Para journeys saltados, usa la misma técnica de flags del ejercicio anterior.

---

**✅ Solución:**
```sql
-- 1) Duplicados exactos
SELECT user_id, event_name, event_ts, COUNT(*) AS n
FROM events
GROUP BY user_id, event_name, event_ts
HAVING COUNT(*) > 1
ORDER BY n DESC;

-- 2) Eventos en el futuro
SELECT *
FROM events
WHERE event_ts > CURRENT_TIMESTAMP;

-- 3) Usuarios con subscribe_premium pero sin play_song
WITH etapas AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'play_song'         THEN 1 ELSE 0 END) AS tiene_play,
        MAX(CASE WHEN event_name = 'subscribe_premium'  THEN 1 ELSE 0 END) AS tiene_premium
    FROM events
    GROUP BY user_id
)
SELECT user_id
FROM etapas
WHERE tiene_premium = 1 AND tiene_play = 0;
```

> **Resultado esperado:** Sin duplicados, sin eventos futuros, y todos los premium pasaron por play_song. Los datos están limpios — podemos confiar en nuestro análisis.

---
# Cierre (5 min)

## Discusión en plenaria

Cada equipo comparta brevemente:
1. ¿Cuál fue el **mayor cuello de botella** del funnel de MusicFlow?
2. ¿Qué **hipótesis de negocio** formularon para explicarlo?
3. ¿Qué **diferencia de retención** encontraron entre canales o cohorts?

## Takeaways de la sesión

1. **Explorar antes de analizar:** Siempre valida la calidad de datos y familiarízate con los eventos antes de construir el funnel.
2. **CTEs son tu mejor herramienta:** Dividen consultas complejas en pasos claros, legibles y debuggeables.
3. **El funnel muestra el "dónde":** Identifica dónde se pierden usuarios. La interpretación dice el "por qué".
4. **Los cohorts muestran el "cuándo":** ¿Las mejoras al producto se reflejan en mejor retención de cohortes nuevas?
5. **Segmentar revela insights:** El funnel general esconde diferencias entre países, canales y dispositivos.

## Siguientes pasos

- **Próxima sesión:** Sprint 4 — A/B Testing y segmentación avanzada en funnels.
- **Tarea:** Exporta la tabla pivote de retención a Google Sheets y crea un heatmap con formato condicional.
- **Desafío extra:** Agrega una 6ta etapa al funnel de MusicFlow: `share_playlist` (compartir playlist). ¿Cómo cambia el embudo?
- **Recordatorio:** La grabación de la sesión y este notebook estarán disponibles para repaso.

## Resumen de ejercicios

| # | Ejercicio | Nivel | Tema | Modalidad |
|---|-----------|-------|------|-----------|
| 0.1 | Eventos únicos | ⭐ | Exploración | Breakout |
| 0.2 | Usuarios activos por mes | ⭐ | Exploración | Breakout |
| 0.3 | Playlist sin suscripción | ⭐⭐ | CTEs + Set diff | Breakout |
| 1.1 | Resumen de actividad | ⭐ | CTEs | Coding Together |
| 1.2 | Suscriptores + país | ⭐ | CTEs + JOIN | Coding Together |
| 1.3 | Pipeline encadenado | ⭐⭐ | CTEs encadenadas | Coding Together |
| 1.4 | Primer evento con ROW_NUMBER | ⭐⭐⭐ | CTEs + Window | Coding Together |
| 2.1 | Funnel completo Q1 | ⭐⭐ | Funnels | Coding Together |
| 2.2 | Funnel febrero | ⭐⭐ | Funnels | Coding Together |
| 2.3 | Tiempos entre etapas | ⭐⭐⭐ | Funnels + latencia | Coding Together |
| 3.1 | Cohorts mensuales | ⭐ | Cohorts | Coding Together |
| 3.2 | Retención semanal | ⭐⭐⭐ | Cohorts | Coding Together |
| 3.3 | Pivote para heatmap | ⭐⭐⭐ | Cohorts + pivote | Coding Together |
| 3.4 | Retención por canal | ⭐⭐⭐ | Cohorts segmentados | Coding Together |
| 4.1 | Journey individual | ⭐⭐ | Integrador | Breakout |
| 4.2 | Funnel por país | ⭐⭐ | Integrador | Breakout |
| 4.3 | Clasificación de usuarios | ⭐⭐⭐ | Integrador | Breakout |
| 4.4 | Checks de calidad | ⭐⭐ | Data quality | Breakout |